In [ ]:
import pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, contextily as cx

# analyses

In [ ]:
data_osm2 = pd.read_csv('data/fig4/contacts_matchdays_osmmapped.csv', low_memory=False)
data_osmpoint2 = pd.read_csv('data/fig4/contacts_matchdays_osmpointmapped2.csv', low_memory=False)

In [ ]:
data_osm2 = data_osm2[data_osm2.city.isin(['Stuttgart','Frankfurt am Main','Dortmund','München'])]
data_osmpoint2 = data_osmpoint2[data_osmpoint2.city.isin(['Stuttgart','Frankfurt am Main','Dortmund','München'])]

In [ ]:
data_osm2_bef = data_osm2[(data_osm2.hour_rel.isin([-2,-1]))].copy() # [(data_osm2.hour_rel < 0)].copy() [(data_osm2.hour_rel.isin([-2,-1]))].copy()
data_osm2_bef['match_rel'] = 'before match'
data_osm2_dur = data_osm2[(data_osm2.hour_rel.isin([0,1]))].copy()
data_osm2_dur['match_rel'] = 'during match'
data_osm2_aft = data_osm2[(data_osm2.hour_rel.isin([2,3]))].copy() # [(data_osm2.hour_rel > 1)].copy() [(data_osm2.hour_rel.isin([2,3]))].copy()
data_osm2_aft['match_rel'] = 'after match'
data_osm2_all = data_osm2.copy()
data_osm2_all['match_rel'] = 'day of the match'
data_osm2 = pd.concat([data_osm2_bef, data_osm2_dur, data_osm2_aft])#, data_osm2_all])

data_osmpoint2_bef = data_osmpoint2[(data_osmpoint2.hour_rel.isin([-2,-1]))].copy() # [(data_osmpoint2.hour_rel < 0)].copy() [(data_osmpoint2.hour_rel.isin([-2,-1]))].copy()
data_osmpoint2_bef['match_rel'] = 'before match'
data_osmpoint2_dur = data_osmpoint2[(data_osmpoint2.hour_rel.isin([0,1]))].copy()
data_osmpoint2_dur['match_rel'] = 'during match'
data_osmpoint2_aft = data_osmpoint2[(data_osmpoint2.hour_rel.isin([2,3]))].copy() # [(data_osmpoint2.hour_rel > 1)].copy() [(data_osmpoint2.hour_rel.isin([2,3]))].copy()
data_osmpoint2_aft['match_rel'] = 'after match'
data_osmpoint2_all = data_osmpoint2.copy()
data_osmpoint2_all['match_rel'] = 'day of the match'
data_osmpoint2 = pd.concat([data_osmpoint2_bef, data_osmpoint2_dur, data_osmpoint2_aft])#, data_osmpoint2_all])

In [ ]:
set(data_osmpoint2[(data_osmpoint2.amenity_point=='fast_food') & (data_osmpoint2.match_rel=='during match') & (data_osmpoint2.city=='Stuttgart')].tl)

In [ ]:
set(data_osm2[(data_osm2.amenity=='parking') & (data_osm2.match_rel=='before match') & (data_osm2.city=='Stuttgart')].tl)

In [ ]:
set(data_osm2[(data_osm2.leisure=='stadium') & (data_osm2.match_rel=='before match') & (data_osm2.city=='Stuttgart')].tl)

In [ ]:
set(data_osm2[(data_osm2.route=='light_rail') & (data_osm2.match_rel=='before match') & (data_osm2.city=='Stuttgart')].tl)

- `amenity='fast_food'`: 601852203230 (city center)
- `amenity='parking'`: 610751303310 (near stadium)
- `leisure='stadium'`: 6107513132(20) (inside stadium)
- `route='light_rail'`: 579593100010 (far from stadium)

In [ ]:
tile_c = 601852203230
gdf = gpd.read_file(f'data/fig4/contact_osm_map_osmid_{tile_c}.gpkg')
gdf

In [ ]:
gdf_sam = gdf.iloc[:len(gdf)//2+1].copy()
gdf_sam.explore(color=gdf_sam.color, marker_kwds={"radius": 10})

In [ ]:
# Reproject to Web Mercator
gdf = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(20,20))  # Wide rectangle

# Plot with basemap
gdf.plot(ax=ax, alpha=0.5, edgecolor='k')
ax.set_aspect('auto')  # Disable equal aspect ratio

# basemap sources: OpenStreetMap.Mapnik CartoDB.Positron CartoDB.DarkMatter Esri.WorldImagery
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.axis(False)

plt.show()

code to visualize contacts with polygons:
```
with t2osm as (
	select t2o.tile_id, tile, unnest(osm_ids) as osm_id
		, st_transform(st_translate(st_setsrid(tile6togeo(t2o.tile), 32632), tx.minx, tx.miny), 3857) as geopoint_t2o
		, st_transform(st_translate(st_setsrid(tile8togeo(610751313220), 32632), tx.minx, tx.miny), 3857) as geopoint_c
	from tuberlin_euro2024_tile2osmid as t2o
	join txc_dt_grid_1000m as tx on t2o.tile_id = tx.tile_id
	where tile = 6107513132
)
select osm.way, tile82polygon(t2osm.geopoint_c), t2osm.*
from t2osm
join tuberlin_euro2024_osm as osm on osm.osm_id = t2osm.osm_id and osm.tile_id = t2osm.tile_id
where osm.leisure='stadium'--osm.barrier='wall'--osm.amenity='parking'--osm.route='light_rail'
```

code to visualize contacts with points:
```
with t2osm as (
	select t2o.tile_id, tile, unnest(osm_ids) as osm_id
		, st_transform(st_translate(st_setsrid(tile6togeo(t2o.tile), 32632), tx.minx, tx.miny), 3857) as geopoint_t2o
		, st_transform(st_translate(st_setsrid(tile8togeo(601852203230), 32632), tx.minx, tx.miny), 3857) as geopoint_c
	from tuberlin_euro2024_tile2osmid as t2o
	join txc_dt_grid_1000m as tx on t2o.tile_id = tx.tile_id
	where tile = 601852203230
)
select poi.way_point, tile82polygon(t2osm.geopoint_c), poi.way_polygon, t2osm.*
from t2osm
join euro2024_poi_all_smallestpolygon as poi on poi.osm_id_polygon = t2osm.osm_id
--where poi.amenity='fast_food'
```